# Artigo 0006 - LLM Evals e Regressao

Notebook com geracao local e avaliacao automatizada usando modelos gratuitos do Hugging Face.

## Candidatos usados
- `qwen_generico`
- `qwen_estruturado`
- `flan_estruturado`

## Modelo de embedding para avaliacao
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

A ideia aqui e comparar candidatos completos e nao apenas uma versao v1 contra uma v2.

In [ ]:
from pathlib import Path
import sys

# Permite executar o notebook tanto a partir da pasta do artigo
# quanto a partir da raiz do repositório sem quebrar os imports locais.
ARTICLE_ROOT = Path.cwd()
if not (ARTICLE_ROOT / "src").exists():
    ARTICLE_ROOT = ARTICLE_ROOT.parent
sys.path.insert(0, str(ARTICLE_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.eval_runner import (
    DEFAULT_CANDIDATES,
    DEFAULT_EMBEDDING_MODEL,
    run_evals,
)

## Estrutura do dataset de avaliação

Diferente do laboratório de prompt, aqui o dataset já nasce orientado a decisão de release. Cada linha informa:

- pergunta;
- contexto oficial;
- gabarito ideal;
- keywords críticas;
- categoria;
- criticidade.

In [ ]:
dataset_preview = pd.read_csv(ARTICLE_ROOT / "data" / "eval_dataset.csv")

# O recorte abaixo deixa claro que a esteira não está olhando só para pergunta e resposta.
# Ela também carrega categoria e criticidade para pesar o impacto real da regressão.
dataset_preview[["pergunta", "categoria", "criticidade"]]

In [ ]:
# Executa a esteira em um recorte pequeno para iteração rápida.
# O runner grava artefatos adicionais que ajudam a entender ranking,
# regressão por caso e comportamento por categoria.
resultados, resumo, relatorio = run_evals(limit=3)
print("Candidatos:", ", ".join(DEFAULT_CANDIDATES))
print("Modelo de embedding:", DEFAULT_EMBEDDING_MODEL)
resumo


## Leitura do ranking

A tabela abaixo já mostra algo importante: não estamos comparando apenas duas versões de prompt. Estamos comparando candidatos completos de inferência, com combinação entre modelo e instrução.

In [ ]:
resultados[["candidate", "modelo_geracao", "pergunta", "score_final", "score_ponderado", "resposta"]]

In [ ]:
comparativo = resultados.pivot(index="pergunta", columns="candidate", values="score_ponderado")
comparativo

In [ ]:
ax = resumo.set_index("candidate")[["score_ponderado", "score_final", "faithfulness"]].plot(
    kind="bar",
    figsize=(10, 4),
    ylim=(0, 1.6),
    title="Comparacao entre candidatos de inferencia local",
)
ax.set_ylabel("score")
plt.xticks(rotation=0)
plt.show()

In [ ]:
print(relatorio)

## Artefatos auxiliares da esteira

Além do ranking principal, o runner escreve arquivos auxiliares para análise operacional:

- `candidate_summary_by_category.csv`
- `critical_cases.csv`
- `regressoes_detectadas.csv`
- `relatorio_evals.md`

In [ ]:
category_summary = pd.read_csv(ARTICLE_ROOT / "data" / "candidate_summary_by_category.csv")
critical_cases = pd.read_csv(ARTICLE_ROOT / "data" / "critical_cases.csv")
regressions = pd.read_csv(ARTICLE_ROOT / "data" / "regressoes_detectadas.csv")

# Desempenho por categoria. Este recorte ajuda a ver se um candidato
# é bom no geral, mas ruim exatamente nos cenários que mais importam.
category_summary

In [ ]:
# Casos críticos e regressões detectadas. Em um pipeline maduro,
# é aqui que o time decide se a troca sobe ou não para produção.
critical_cases, regressions